# Produttivita e reddito: API

Notebook sulla produttivita italiana costruito solo da API ufficiali: World Bank per PIL, PIL per occupato, investimenti e produzione; Eurostat per produttivita oraria, produttivita per occupato, costo del lavoro per unita di prodotto e ore annue per occupato.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "analisi").exists():
    ROOT = ROOT.parent
if not (ROOT / "analisi").exists():
    raise RuntimeError("Esegui il notebook dalla root del repo o da una sottocartella del repo.")
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

from analisi.utils.api_indicators import run_decline_analysis, plot_indicator

# I dati vengono richiesti alle API ufficiali a ogni esecuzione.
REFRESH_API = True

result = run_decline_analysis(refresh=REFRESH_API)
panel = result["panel"]
summary = result["summary"]
transforms = result["transforms"]


## Indicatori di produttivita e reddito

La tabella mette insieme livello piu recente, variazione dal baseline e gap rispetto ai peer.


In [ ]:
prod_themes = ["produttivita_reddito", "produzione", "investimenti"]
prod_summary = summary[summary["theme"].isin(prod_themes)].copy()
prod_summary[[
    "theme", "indicator_label", "unit", "latest_year", "latest_value_italy",
    "change_abs_since_baseline", "gap_vs_peer_mean", "source"
]].sort_values(["theme", "indicator_label"])


## Indici e crescita decennale

Gli indici usano il primo dato disponibile dal 2000 come baseline; la crescita decennale e annualizzata quando la serie lo consente.


In [ ]:
focus = [
    "NY.GDP.PCAP.KD",
    "SL.GDP.PCAP.EM.KD",
    "ESTAT_RLPR_HW_I15",
    "ESTAT_RLPR_PER_I15",
    "ESTAT_HW_EMP",
    "ESTAT_NULC_HW_I15",
]
transforms[
    (transforms["indicator_id"].isin(focus)) & (transforms["country_key"].isin(["ITA", "DEU", "FRA", "ESP", "EU"]))
][[
    "indicator_label", "country", "year", "value", "index_from_baseline", "growth_10y_annualized_pct"
]].sort_values(["indicator_label", "country", "year"]).tail(30)


## Grafici


In [ ]:
for indicator_id in focus:
    display(Image(filename=str(plot_indicator(panel, indicator_id))))
